# Phase 2: Feature Engineering

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_DIR, 'data')):
    REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)

TRAIN = {l1: os.path.join(REPO_DIR, f'data/train/{l1}/kvl_shared_task_{l1}_train.csv') for l1 in ['es','de','cn']}
DEV = {l1: os.path.join(REPO_DIR, f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv') for l1 in ['es','de','cn']}

In [ ]:
from features.pipeline import FeaturePipeline

for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    pipeline = FeaturePipeline(l1=l1, frozen_features_path=None, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipeline.fit(train_df)
    out = pipeline.transform(train_df)
    print(f'L1={l1} feature matrix shape: {out.shape}')
    print('Feature names:', pipeline.get_feature_names())
    print(out.head(3))
    print()

In [ ]:
if pipeline._freq_norms:
    for name, tbl in getattr(pipeline._freq_norms, '_tables', {}).items():
        print(f'Resource {name}: rows={len(tbl)}')

In [ ]:
from scipy.stats import pearsonr

summary = []
for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    pipeline = FeaturePipeline(l1=l1, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipeline.fit(train_df)
    out = pipeline.transform(train_df)
    if 'GLMM_score' not in out.columns:
        continue
    y = out['GLMM_score']
    for col in out.columns:
        if col == 'GLMM_score': continue
        r, _ = pearsonr(out[col].fillna(out[col].median()), y)
        summary.append({'L1': l1, 'feature': col, 'r': r})
sum_df = pd.DataFrame(summary)
pivot = sum_df.pivot(index='feature', columns='L1', values='r')
print(pivot.round(4))

In [ ]:
print('Direction check: edit_distance should have NEGATIVE r for es/de (cognate=easy=high score).')
print('reveal_ratio should have POSITIVE r (more clue = easier).')
print(pivot.loc[['edit_distance','reveal_ratio']].round(4) if 'edit_distance' in pivot.index else 'edit_distance not in pivot')